In [1]:
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from criteria.multi_criteria import metoda_vazeneho_suctu, metoda_topsis, metoda_poradia_varianty

In [2]:
def poradie(scores, reverse=False):
    if reverse:
        return np.argsort(np.argsort(scores)) + 1
    else:
        return np.argsort(np.argsort(-np.array(scores))) + 1

In [3]:
# 20 najlepších ubytovaní v Hamburgu s filtrami (Best reviewed & lowest price)

data = [
        {"name": "Central City Apartment for 8 in Hamburg", "rating": 10.0, "price": 245},
        {"name": "Little Wizard Apartment Hamburg", "rating": 10.0, "price": 290},
        {"name": "Holiday apartment in Bramfelder Chaussee 312- 2", "rating": 10.0, "price": 314},
        {"name": "Holiday apartment in Bramfelder Chaussee 312- 1", "rating": 10.0, "price": 316},
        {"name": "Modern 3-Bedroom near Alster Hamburg", "rating": 10.0, "price": 325},
        {"name": "Magisches Schlafzimmer", "rating": 10.0, "price": 486},
        {"name": "Billard Tischtennis Tischkicker", "rating": 10.0, "price": 500},
        {"name": "Stilvolles Loft in Hamburg-Uhlenhorst", "rating": 9.5, "price": 205},
        {"name": "Apartment 41B", "rating": 9.5, "price": 207},
        {"name": "Berner Bude 1", "rating": 9.5, "price": 209},
        {"name": "Luxury Apartment with 3BR", "rating": 9.8, "price": 218},
        {"name": "Residence Reeperbahn Harbour & Schanze", "rating": 9.5, "price": 254},
        {"name": "Das ELBCOTTAGE", "rating": 9.8, "price": 280},
        {"name": "Residence Hamburg - Reeperbahn - St Pauli", "rating": 9.7, "price": 284},
        {"name": "Peaceful City Garden Apartment", "rating": 9.5, "price": 328},
        {"name": "Große Fewo mit 2 SZ & 2 Bäder", "rating": 9.6, "price": 350},
        {"name": "Studio Pretty", "rating": 9.7, "price": 350},
        {"name": "FOUNDRY - Business & Content Loft", "rating": 9.5, "price": 406},
        {"name": "XXL Family & Friends Duplex nähe Messe", "rating": 9.7, "price": 455},
        {"name": "R B Apartment Hamburg Family - zentrumsnah", "rating": 9.5, "price": 545}
    ]

In [4]:
df = pd.DataFrame(data)
    
df['Booking_Rank'] = range(1, len(df) + 1)
print(df[['name','price','rating','Booking_Rank']].to_string(index=False))

                                           name  price  rating  Booking_Rank
        Central City Apartment for 8 in Hamburg    245    10.0             1
                Little Wizard Apartment Hamburg    290    10.0             2
Holiday apartment in Bramfelder Chaussee 312- 2    314    10.0             3
Holiday apartment in Bramfelder Chaussee 312- 1    316    10.0             4
           Modern 3-Bedroom near Alster Hamburg    325    10.0             5
                         Magisches Schlafzimmer    486    10.0             6
                Billard Tischtennis Tischkicker    500    10.0             7
          Stilvolles Loft in Hamburg-Uhlenhorst    205     9.5             8
                                  Apartment 41B    207     9.5             9
                                  Berner Bude 1    209     9.5            10
                      Luxury Apartment with 3BR    218     9.8            11
         Residence Reeperbahn Harbour & Schanze    254     9.5            12

In [5]:
# Príprava rozhodovacej matice. Nastavenie optimalizácie:
# cena sa minimalizuje (False), hodnotenie sa maximalizuje (True)

matrix = df[['price', 'rating']].values
maximization = [False, True]
weight_price = 0.4
weight_rating = 0.6
weights = [weight_price, weight_rating]

In [6]:
_, topsis_scores, _, _ = metoda_topsis(matrix, weights, maximization)
df['TOPSIS_Rank'] = poradie(topsis_scores, reverse=False)

print(df[['name', 'Booking_Rank', 'TOPSIS_Rank','price','rating']].to_string(index=False))

                                           name  Booking_Rank  TOPSIS_Rank  price  rating
        Central City Apartment for 8 in Hamburg             1            5    245    10.0
                Little Wizard Apartment Hamburg             2            9    290    10.0
Holiday apartment in Bramfelder Chaussee 312- 2             3           10    314    10.0
Holiday apartment in Bramfelder Chaussee 312- 1             4           11    316    10.0
           Modern 3-Bedroom near Alster Hamburg             5           12    325    10.0
                         Magisches Schlafzimmer             6           18    486    10.0
                Billard Tischtennis Tischkicker             7           19    500    10.0
          Stilvolles Loft in Hamburg-Uhlenhorst             8            2    205     9.5
                                  Apartment 41B             9            3    207     9.5
                                  Berner Bude 1            10            4    209     9.5
          

**TOPSIS generuje úplne iné poradie ako Booking. Hľadá ideálny kompromis (najlepší pomer cena/rating) a prísne penalizuje veľmi drahé ubytovania, aj keď majú hodnotenie 10.0.**

In [7]:
_, _, rank_scores = metoda_poradia_varianty(matrix, weights, maximization)
df['Rank_Method_Rank'] = poradie(rank_scores, reverse=True)


print(df[['name', 'Booking_Rank', 'Rank_Method_Rank','price','rating']].to_string(index=False))

                                           name  Booking_Rank  Rank_Method_Rank  price  rating
        Central City Apartment for 8 in Hamburg             1                 1    245    10.0
                Little Wizard Apartment Hamburg             2                 2    290    10.0
Holiday apartment in Bramfelder Chaussee 312- 2             3                 3    314    10.0
Holiday apartment in Bramfelder Chaussee 312- 1             4                 5    316    10.0
           Modern 3-Bedroom near Alster Hamburg             5                 6    325    10.0
                         Magisches Schlafzimmer             6                 8    486    10.0
                Billard Tischtennis Tischkicker             7                10    500    10.0
          Stilvolles Loft in Hamburg-Uhlenhorst             8                11    205     9.5
                                  Apartment 41B             9                12    207     9.5
                                  Berner Bude 1   

**Poskytuje kompromisné poradie, ktoré vyhladzuje extrémy. Výsledky sa stále líšia od Bookingu, no ukazujú iný pohľad na preferencie.**

In [9]:
_, _, wsm_scores = metoda_vazeneho_suctu(matrix, weights, maximization)
df['WSM_Rank'] = poradie(wsm_scores, reverse=False)

print(df[['name', 'Booking_Rank', 'WSM_Rank','price','rating']].to_string(index=False))

                                           name  Booking_Rank  WSM_Rank  price  rating
        Central City Apartment for 8 in Hamburg             1         1    245    10.0
                Little Wizard Apartment Hamburg             2         2    290    10.0
Holiday apartment in Bramfelder Chaussee 312- 2             3         3    314    10.0
Holiday apartment in Bramfelder Chaussee 312- 1             4         4    316    10.0
           Modern 3-Bedroom near Alster Hamburg             5         5    325    10.0
                         Magisches Schlafzimmer             6         8    486    10.0
                Billard Tischtennis Tischkicker             7         9    500    10.0
          Stilvolles Loft in Hamburg-Uhlenhorst             8        12    205     9.5
                                  Apartment 41B             9        13    207     9.5
                                  Berner Bude 1            10        14    209     9.5
                      Luxury Apartment with

**Pri tomto nastavení váh sa WSM najviac približuje reálnemu poradiu na Bookingu. Prvých päť ubytovania sa presne zhodovalo**